# 02 — Data Cleaning
### Business Request: "Prove this data is trustworthy before we analyze it."

**Cleaning data vs. changing data.** Cleaning fixes genuine defects — wrong
types, inconsistent formats, stray whitespace — without altering what the
business actually recorded. Changing data means altering real values, and
every such change needs an explicit, documented justification. This
notebook only does the former: every step below is justified by a check
performed immediately before it.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from clean_data import (
    check_missing_values, check_duplicates, check_date_formats,
    check_price_consistency, check_negative_or_zero,
    clean_pizza_sales, engineer_features,
)

df = pd.read_csv('../data/raw/pizza_sales.csv')
df.shape

(48620, 12)

## 1. Data Quality Checklist

Never trust a dataset just because it loaded successfully. Work through a
checklist every time, on every new file:
- missing values
- duplicate records
- inconsistent categories / invalid values
- incorrect data types
- impossible dates / suspicious values
- inconsistent naming

In [2]:
check_missing_values(df)

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

Zero missing values in every column. Good — but that alone doesn't mean the file is clean, as the next check shows.

In [3]:
print("Duplicate pizza_id rows:", check_duplicates(df))
print("Fully duplicate rows:", df.duplicated().sum())

Duplicate pizza_id rows: 0
Fully duplicate rows: 0


### The `order_date` problem

Let's actually look at a sample of raw date values instead of assuming
`pd.to_datetime` will "just work":

In [4]:
df['order_date'].sample(10, random_state=1).tolist()

['27-03-2015',
 '6/12/2015',
 '7/3/2015',
 '21-10-2015',
 '16-04-2015',
 '16-01-2015',
 '5/12/2015',
 '1/10/2015',
 '3/4/2015',
 '1/6/2015']

Notice two different separators: `/` and `-`. Let's quantify it:

In [5]:
check_date_formats(df)

slash_format (M/D/YYYY)    19587
dash_format (D-M-YYYY)     29033
dtype: int64

**This is a real, verified data-quality issue**: 19,587 rows use `M/D/YYYY`
and 29,033 use `D-M-YYYY`, interleaved throughout the file (see
`data/README.md` for the full investigation — the two formats are not a
clean batch cutover by `order_id`). Try the naive approach and watch it fail:

In [6]:
try:
    pd.to_datetime(df['order_date'], format='%m/%d/%Y')
except ValueError as e:
    print(f"Failed as expected: {e}")

Failed as expected: time data "13-01-2015" doesn't match format "%m/%d/%Y". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


### Other checks: price consistency and invalid values

In [7]:
print("total_price mismatches (quantity * unit_price):", check_price_consistency(df))
print()
print(check_negative_or_zero(df, ['quantity', 'unit_price', 'total_price']))

total_price mismatches (quantity * unit_price): 0

quantity       0
unit_price     0
total_price    0
dtype: int64


## 2. Cleaning

`clean_pizza_sales()` (in `src/clean_data.py`) applies exactly the steps
justified above — parse the mixed dates correctly, fix dtypes, strip
whitespace defensively, drop any true duplicates. Read the function's
source before running it, so you know precisely what it does and why.

In [8]:
clean_df = clean_pizza_sales(df)
clean_df.dtypes

pizza_id                      int64
order_id                      int64
pizza_name_id                   str
quantity                      int64
order_date           datetime64[ns]
order_time                   object
unit_price                  float64
total_price                 float64
pizza_size                      str
pizza_category                  str
pizza_ingredients               str
pizza_name                      str
dtype: object

In [9]:
# Confirm the fix: every date is now a real date, and the range makes sense
print(clean_df['order_date'].min(), '->', clean_df['order_date'].max())
print("Any parsing failures (NaT)?", clean_df['order_date'].isnull().sum())

2015-01-01 00:00:00 -> 2015-12-31 00:00:00
Any parsing failures (NaT)? 0


## 3. Feature Engineering

Only features that make analytical sense given the columns we actually
have (`data/README.md` — no invented columns). `engineer_features()` adds:

| Column | Why |
|---|---|
| `order_month`, `order_day_name` | needed for every time-trend question in Notebook 03/04 |
| `order_hour`, `time_of_day` | needed for "when do customers order?" |
| `is_weekend` | needed for weekday vs. weekend comparisons |
| `price_band` | needed for "are cheap pizzas driving volume?" style questions |


In [10]:
final_df = engineer_features(clean_df)
final_df[['order_date','order_month','order_day_name','order_hour','time_of_day','is_weekend','price_band']].head()

,order_date,order_month,order_day_name,order_hour,time_of_day,is_weekend,price_band
0,2015-01-01,January,Thursday,11,Lunch,False,Standard
1,2015-01-01,January,Thursday,11,Lunch,False,Standard
2,2015-01-01,January,Thursday,11,Lunch,False,Premium
3,2015-01-01,January,Thursday,11,Lunch,False,Premium
4,2015-01-01,January,Thursday,11,Lunch,False,Standard


## Your Turn

1. How many orders fall on a weekend (`is_weekend == True`) vs. a weekday? Does that match your intuition for a pizza restaurant?
2. Look at the `price_band` value counts. Which band has the most SKUs?
3. **Challenge:** the cutoffs for `price_band` (`$12.50` / `$17.50`) are hardcoded in `src/clean_data.py`. Recompute them using `unit_price.quantile([0.33, 0.66])` instead, and discuss: would data-driven cutoffs change the business story?

In [11]:
# Your code here


## Save the cleaned, feature-engineered file

This matches what `python src/clean_data.py` produces from the command
line — running it here just makes the transformation visible step by step.

In [12]:
final_df.to_csv('../data/processed/pizza_sales_clean.csv', index=False)
print(f"Saved {len(final_df):,} rows, {len(final_df.columns)} columns.")

Saved 48,620 rows, 18 columns.


## What we learned

- The dataset's one real defect (mixed date formats) is now fixed, with the fix documented and justified — not silently patched.
- No values were changed, only their *representation* (text → date, float → int).
- We added 6 new analytical columns without inventing any new source data.

**Next:** `03_eda.ipynb` switches to querying the (already-clean) database via `src/extract_data.py` and starts asking real business questions.